In [ ]:
import json
import time
import re
import copy
from pathlib import Path

In [ ]:
# Import kaggle_benchmarks as kbench for using free AI quota provided by Kaggle (10 usd/day - 100 usd/month). 
# This is a temporary solution to avoid the cost of using API as OpenAI.
import kaggle_benchmarks as kbench

# My project uses the "google/gemini-3.6-flash" provided by Kaggle Benchmarks.
llm=kbench.llms["google/gemini-3.6-flash"]

#  You can check the available LLMs by printing the keys of kbench.llms dictionary.
print(list(kbench.llms.keys())) 

In [ ]:
# Call the LLM many times can lead to exceeding the free quota or overloading the LLM,
# so you should consider splitting the files into parts and adding a delay between calls.

PART = 1
TOTAL_PARTS = 1
DELAY_BETWEEN_CALLS = 60.0

def get_total_files(transcript_dir, part=PART, total_parts=TOTAL_PARTS):
    raw_paths = transcript_dir.glob("*_transcript.json")
    already_grouped_paths = transcript_dir.glob("*_transcript_event.json")
    all_json_paths = sorted(set(
        [str(p) for p in raw_paths] + [str(p) for p in already_grouped_paths]
    ))
    total_all_files = len(all_json_paths)

    if total_all_files == 0:
        print(f"⚠️ Không tìm thấy file JSON nào trong {transcript_dir}")

    # Chunking
    avg_chunk = total_all_files // total_parts
    start_idx = (part - 1) * avg_chunk
    end_idx = total_all_files if part == total_parts else part * avg_chunk

    part_paths = all_json_paths[start_idx:end_idx]
    total_part_files = len(part_paths)

    print(f"🔥 PART {part}/{total_parts} - Transcript dir: {transcript_dir}")
    print(f"📊 Xử lý từ file #{start_idx + 1} đến #{end_idx} (Tổng {total_part_files}/{total_all_files} files)")
    print("----------------------------------------------------------------------")

    return part_paths

def build_prompt(texts):
    payload = "\n".join(texts)
    prompt = "" 
    # for accuracy, building the prompt should be done carefully, and it can be based on the topic of the video. 
    # Check benchmark/prompts for details. 
    # REQUIRED: You must provide a list of sequences 'texts' and processed 'payload' to build the prompt. 
    # DO NOT leave the list empty,
    return prompt

In [ ]:
transcript_dir = Path("") #folder containing all transcript files
base_output_dir = Path("") #folder to save the output files

@kbench.task(name=f"Event Grouping",  description=f"Gom lại segments thành events đúng theo chủ đề")
def fix_event_grouping_task(llm) -> None:
    part_paths = get_total_files(transcript_dir)
    
    for idx, input_path in enumerate(part_paths, 1):
        filename = Path(input_path).name
        output_path = base_output_dir / filename

        print(f"🎬 [{idx}/{len(part_paths)}] Đang xử lý: {filename}")
        try:
            with open(input_path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"    ❌ Lỗi đọc file: {e}. Bỏ qua file này.")
            continue
    
        video_key = data.get('video', Path(filename).stem)

        if output_path.resolve() == Path(input_path).resolve():
            raise RuntimeError(f"output_path trùng input_path ({input_path}), dừng để tránh ghi đè file gốc!")

        def clean_text(t):
            return re.sub(r"\s+", " ", t).strip()
        
        flat_segments = [f"[{i}] {clean_text(s['text'])}" for i, s in enumerate(data['segments'])]
        total_segs = len(flat_segments)
        
        if total_segs == 0:
            print("    ⏩ File không có segments. Ghi kết quả rỗng.")
            output_path.parent.mkdir(parents=True, exist_ok=True)
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump({video_key: {"group_status": "empty", "events": []}}, f, ensure_ascii=False, indent=2)
            continue

        prompt = build_prompt(flat_segments)

        max_retries = 3
        retry_count = 0
        success = False
        built_events = []

        while retry_count < max_retries and not success:
            try:
                response_text = llm.prompt(prompt)

                cleaned = re.sub(
                    r"^```(?:json)?|```$", "", response_text.strip(),
                    flags=re.MULTILINE | re.IGNORECASE
                ).strip()
                groups = json.loads(cleaned)

                # VALIDATE
                flat_ids = [i for grp in groups for i in grp]
                if flat_ids != list(range(total_segs)):
                    raise ValueError(
                        f"Validate thất bại: AI trả về {len(flat_ids)} chỉ số, không khớp {total_segs} câu gốc."
                    )

                # REBUILD EVENTS
                events = []
                for grp in groups:
                    segs_in_group = [copy.deepcopy(data['segments'][i]) for i in grp]
                    events.append({
                        "seg_lst": [s["id"] for s in segs_in_group],
                        "text_lst": [s["text"] for s in segs_in_group],
                    })

                built_events = events
                success = True
                time.sleep(DELAY_BETWEEN_CALLS)

            except Exception as e:
                retry_count += 1
                print(f"    ⚠️ Lỗi file {filename} (Lần thử {retry_count}/{max_retries}): {e}")
                if retry_count < max_retries:
                    backoff = retry_count * 10.0
                    print(f"    💤 Nghỉ {backoff}s rồi thử lại...")
                    time.sleep(backoff)
                else:
                    print("    ❌ FAILED!")

        output_data = {
            video_key: {
                "group_status": "success" if success else "failed",
                "events": built_events,
            }
        }

        output_path.parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(output_data, f, ensure_ascii=False, indent=2)

        if success:
            if len(built_events) > 10:
                print(f"    ⚠️ Đã xử lý xong: {len(built_events)} events -> {output_path}")
            else:
                print(f"    ✅ Đã xử lý xong: {len(built_events)} events -> {output_path}")
        else:
            print(f"    ⚠️ File {filename} thất bại, đã ghi status=failed (events rỗng) -> {output_path}")

    print(f"\n⚔️ HOÀN THÀNH: PART {PART}/{TOTAL_PARTS}!")
    kbench.assertions.assert_true(True, expectation=f"Done Part {PART}")

In [ ]:
# Run the task with the specified LLM
fix_event_grouping_task.run(llm=llm)